# Part e): Classification of handwritten digits

We classify the handwritten digits of the MNIST data set with the feed-forward network from the
regression notebook. Only the output layer and the cost function change: the network ends in
ten logits, and the softmax and the categorical cross entropy turn them into class
probabilities.

The notebook is structured as follows:

1. **Data.** The canonical split into 60 000 training and 10 000 test images, with a validation
   set taken from the training images. All tuning uses the validation set.
2. **Baselines.** Always predicting the most common digit, and logistic regression, the
   network with no hidden layers. A network with hidden layers is only worth its cost if it
   beats logistic regression.
3. **Neural networks.** A search over the activation function, depth, width and learning rate,
   then over the learning rate and the $\ell_2$ penalty for the best architecture.
4. **Comparison with Scikit-Learn** for both models.
5. **Test set.** The selected models are evaluated once on the test set.

> **LLM note:** Most of this notebook was written or rewritten with Claude assistance (level 4); the main cells are marked below. The experiment design and the search grids were chosen by the author. See the appendix of the report for the full declaration.

In [ ]:
import hashlib
import warnings
from collections.abc import Callable
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scienceplots  # noqa: F401
from fys_stk4155_p1.optimization.optimizers import Adam, Optimizer
from matplotlib.colors import Normalize, TwoSlopeNorm
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression as SklearnLogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from tensorflow.keras.datasets import mnist

from fys_stk4155_p2.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from fys_stk4155_p2.models.logistic import LogisticRegression
from fys_stk4155_p2.nn.activations import Activation, LeakyReLU, ReLU, Sigmoid
from fys_stk4155_p2.nn.layers import Dense, Layer, Linear
from fys_stk4155_p2.nn.losses import CategoricalCrossEntropy
from fys_stk4155_p2.nn.network import FFNN, Schedule

plt.style.use(["science", "ieee"])

## Configuration

All settings are collected here. The searches are slow, so their results are cached in
`RESULTS_DIR`, in files named by a hash of the settings: changing a setting runs the search
again instead of loading stale results.

In [ ]:
FIGURE_DIR = Path("../docs/figures/classification")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
# Cached results of the slow hyperparameter searches (git-ignored)
RESULTS_DIR = Path("../data/classification")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
N_CLASSES = 10
VALIDATION_SIZE = 10_000

# Shared by all models
OPTIMIZER = Adam
BATCH_SIZE = 128

# Logistic regression baseline, trained for a fixed number of epochs
LOGREG_EPOCHS = 50
LOGREG_LEARNING_RATE = [1e-4, 1e-3, 1e-2, 1e-1]
LOGREG_REGULARIZATION_STRENGTH = [0.0, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2]

# Neural networks, trained with early stopping and a decaying learning rate
ACTIVATION = [Sigmoid, ReLU, LeakyReLU]
N_HIDDEN_LAYERS = [4, 8, 12]
N_NODES_PER_LAYER = [20, 40, 80]
N_LEARNING_RATE = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1]
REGULARIZATION_STRENGTH = [0.1, 0.01, 0.001, 0.0001, 0.0]
MAX_EPOCHS = 50  # upper limit, as early stopping usually ends training sooner
PATIENCE = 5  # epochs without a lower validation cost before stopping
LR_SCHEDULE = "exponential"
LR_DECAY_FACTOR = 10  # the learning rate falls by this factor over MAX_EPOCHS

# Accuracies below this are drawn in the lightest color in the network heat maps, so that
# the failed runs do not wash out the differences between the good ones
HEATMAP_FLOOR = 90.0

## Data

The images are flattened to 784 features and scaled to $[0, 1]$. We set aside a stratified
validation set of `VALIDATION_SIZE` training images for all tuning; the test set is left
untouched until the final evaluation. The networks take one-hot encoded targets.

In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

x_train = x_train.reshape(len(x_train), -1) / 255.0
x_test = x_test.reshape(len(x_test), -1) / 255.0

x_fit, x_val, y_fit, y_val = train_test_split(
    x_train, y_train, test_size=VALIDATION_SIZE, stratify=y_train, random_state=SEED
)
Y_fit = np.eye(N_CLASSES)[y_fit]
Y_val = np.eye(N_CLASSES)[y_val]

print(f"Training data shape: {x_fit.shape}, validation data shape: {x_val.shape}")

## Helper functions

Caching of the searches, the per-digit metrics, and the heat maps and confusion matrices used
for both models.

> **LLM note:** The following cell was written with Claude assistance (level 4).

In [ ]:
def cached_search(
    name: str, settings: dict[str, object], search: Callable[[], pd.DataFrame]
) -> pd.DataFrame:
    """Load the results of a search from RESULTS_DIR, or run and save it.

    The file name includes a hash of the settings, so changing the configuration runs
    the search again instead of loading stale results.
    """
    key = hashlib.sha1(repr(sorted(settings.items())).encode()).hexdigest()[:8]
    results_path = RESULTS_DIR / f"{name}_{key}.csv"
    if results_path.exists():
        print(f"Loaded cached results from {results_path}")
        return pd.read_csv(results_path)
    results = search()
    results.to_csv(results_path, index=False)
    return results


def class_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> pd.DataFrame:
    """Precision, recall and F1 score of each digit."""
    return pd.DataFrame(
        {
            "precision": precision_score(y_true, y_pred, n_classes=N_CLASSES),
            "recall": recall_score(y_true, y_pred, n_classes=N_CLASSES),
            "f1": f1_score(y_true, y_pred, n_classes=N_CLASSES),
        }
    ).rename_axis("digit")


def macro_f1(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """F1 score averaged over the digits with equal weights."""
    return float(np.mean(f1_score(y_true, y_pred, n_classes=N_CLASSES)))


def power_label(value: float) -> str:
    """Tick label for a power of ten, or 0."""
    return "0" if value == 0 else f"$10^{{{round(np.log10(value))}}}$"


def accuracy_norm(table: np.ndarray, floor: float | None = HEATMAP_FLOOR) -> Normalize:
    """Color scale for accuracies in percent, clipped from below at ``floor`` if given."""
    if floor is None:
        return Normalize(vmin=table.min(), vmax=table.max())
    return Normalize(vmin=max(floor, table.min()), vmax=table.max(), clip=True)


def style_heatmap(ax: plt.Axes) -> None:
    """Remove ticks, minor ticks and spines from a heat map."""
    ax.minorticks_off()
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)


def plot_heatmap(
    table: np.ndarray,
    x_labels: list[str],
    y_labels: list[str],
    xlabel: str,
    ylabel: str | None,
    filename: str,
    figsize: tuple[float, float],
    floor: float | None = HEATMAP_FLOOR,
    colorbar_label: str = "Validation accuracy (percent)",
) -> None:
    """Heat map of accuracies in percent, with the value written in each cell."""
    norm = accuracy_norm(table, floor)
    fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)
    image = ax.imshow(table, cmap="Blues", norm=norm, aspect="auto")
    for (i, j), value in np.ndenumerate(table):
        color = "white" if norm(value) > 0.6 else "#1f1e1d"
        ax.text(j, i, f"{value:.1f}", ha="center", va="center", fontsize=7, color=color)
    ax.set_xticks(range(len(x_labels)), x_labels)
    ax.set_yticks(range(len(y_labels)), y_labels)
    ax.set_xlabel(xlabel)
    if ylabel is not None:
        ax.set_ylabel(ylabel)
    style_heatmap(ax)
    extend = "neither" if floor is None else "min"
    fig.colorbar(image, ax=ax, label=colorbar_label, shrink=0.9, extend=extend)
    fig.savefig(FIGURE_DIR / filename)
    plt.show()


def plot_confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray, filename: str) -> None:
    """Plot the confusion matrix normalized by row, so the diagonal is each digit's recall."""
    cm = confusion_matrix(y_true, y_pred, n_classes=N_CLASSES)
    cm_norm = cm / cm.sum(axis=1, keepdims=True)

    fig, ax = plt.subplots(figsize=(3.3, 3.0), constrained_layout=True)
    image = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    for (i, j), value in np.ndenumerate(cm_norm):
        if value >= 0.005:
            color = "white" if value > 0.6 else "#1f1e1d"
            ax.text(j, i, f"{100 * value:.0f}", ha="center", va="center", fontsize=5, color=color)
    ax.set_xticks(range(N_CLASSES))
    ax.set_yticks(range(N_CLASSES))
    ax.set_xlabel("Predicted digit")
    ax.set_ylabel("True digit")
    style_heatmap(ax)
    fig.colorbar(image, ax=ax, label="Share of true digit", shrink=0.8)
    fig.savefig(FIGURE_DIR / filename)
    plt.show()


def plot_learning_curves(
    curves: dict[str, tuple[list[float], list[float]]],
    filename: str,
    stopping_epoch: int | None = None,
    log_scale: bool = False,
) -> None:
    """Training (dashed) and validation (solid) cost per epoch, one color per run."""
    colors = ["#1f77b4", "#d62728"]
    fig, ax = plt.subplots(figsize=(3.3, 2.2))
    for (label, (training, validation)), color in zip(curves.items(), colors, strict=False):
        epochs = np.arange(1, len(training) + 1)
        prefix = f"{label}, " if label else ""
        ax.plot(epochs, training, color=color, linestyle="--", label=f"{prefix}training")
        ax.plot(epochs, validation, color=color, linestyle="-", label=f"{prefix}validation")
    if stopping_epoch is not None:
        ax.axvline(stopping_epoch, color="#5f5e5a", linestyle=":", label="Early stopping")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(r"Cross entropy + $\ell_2$ penalty")
    if log_scale:
        ax.set_yscale("log")
    ax.legend(fontsize=5 if len(curves) > 1 else None)
    fig.savefig(FIGURE_DIR / filename)
    plt.show()

## Baselines

### Majority class

The simplest reference is to always predict the most common digit of the training data.

In [ ]:
majority_class = np.bincount(y_fit).argmax()
majority_accuracy = accuracy_score(y_val, np.full_like(y_val, majority_class))
print(f"Majority-class ({majority_class}) validation accuracy: {majority_accuracy:.4f}")

### Logistic regression

Logistic regression is the network with no hidden layers: one Dense layer followed by the
softmax and the cross entropy. We search a grid of learning rates $\gamma$ and $\ell_2$
penalties $\lambda$, scored by the validation accuracy. The penalty multiplies the sum over all
$784 \times 10$ squared weights, while the cross entropy is a mean over the samples, so the
useful values of $\lambda$ are small.

> **LLM note:** The following cell was written with Claude assistance (level 4).

In [ ]:
def fit_logistic_regression(
    learning_rate: float,
    lambda_: float,
    validation_data: tuple[np.ndarray, np.ndarray] | None = None,
) -> tuple[LogisticRegression, list[float]]:
    """Train logistic regression with Adam for LOGREG_EPOCHS epochs on the training split."""
    model = LogisticRegression(
        optimizer=OPTIMIZER(learning_rate=learning_rate),
        regularization="l2",
        lambda_=lambda_,
        seed=SEED,
    )
    history = model.fit(
        X=x_fit,
        y=y_fit,
        epochs=LOGREG_EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=validation_data,
    )
    return model, history


def logreg_search() -> pd.DataFrame:
    rows = []
    for learning_rate, lambda_ in product(LOGREG_LEARNING_RATE, LOGREG_REGULARIZATION_STRENGTH):
        model, _ = fit_logistic_regression(learning_rate, lambda_)
        rows.append(
            {
                "learning_rate": learning_rate,
                "lambda": lambda_,
                "val_accuracy": accuracy_score(y_val, model.predict(x_val)),
            }
        )
    return pd.DataFrame(rows)


logreg_grid = cached_search(
    "logreg",
    {
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "epochs": LOGREG_EPOCHS,
        "learning_rates": LOGREG_LEARNING_RATE,
        "lambdas": LOGREG_REGULARIZATION_STRENGTH,
    },
    logreg_search,
)

best = logreg_grid.loc[logreg_grid["val_accuracy"].idxmax()]
LOGREG_BEST_LEARNING_RATE, LOGREG_BEST_LAMBDA = best["learning_rate"], best["lambda"]
print(
    f"Best: gamma = {LOGREG_BEST_LEARNING_RATE:g}, lambda = {LOGREG_BEST_LAMBDA:g}, "
    f"validation accuracy = {best['val_accuracy']:.4f}"
)

In [ ]:
plot_heatmap(
    100
    * logreg_grid.pivot_table(index="lambda", columns="learning_rate", values="val_accuracy")
    .loc[LOGREG_REGULARIZATION_STRENGTH, LOGREG_LEARNING_RATE]
    .to_numpy(),
    x_labels=[power_label(v) for v in LOGREG_LEARNING_RATE],
    y_labels=[power_label(v) for v in LOGREG_REGULARIZATION_STRENGTH],
    xlabel=r"Learning rate $\gamma$",
    ylabel=r"$\ell_2$ penalty $\lambda$",
    filename="logreg_hyperparameters.pdf",
    figsize=(3.3, 2.6),
    floor=None,
)

#### The selected model

We retrain the best configuration, now tracking the validation cost after every epoch, and
look at the precision, recall and $F_1$ score of each digit.

In [ ]:
logistic_regression, logreg_history = fit_logistic_regression(
    LOGREG_BEST_LEARNING_RATE, LOGREG_BEST_LAMBDA, validation_data=(x_val, y_val)
)
plot_learning_curves(
    {"": (logreg_history, logistic_regression.validation_history)}, "logreg_learning_curve.pdf"
)

logreg_pred = logistic_regression.predict(x_val)
logreg_metrics = class_metrics(y_val, logreg_pred)
print(f"Validation accuracy: {accuracy_score(y_val, logreg_pred):.4f}")
print(f"Macro-averaged F1 score: {logreg_metrics['f1'].mean():.4f}")
logreg_metrics.round(3)

Each column of the weight matrix is a $28 \times 28$ template that the model matches against
the image; red pixels push towards the digit and blue pixels away from it.

In [ ]:
weights = logistic_regression.network.layers[0].weights  # shape (784, 10)
limit = np.abs(weights).max()

fig, axes = plt.subplots(1, N_CLASSES, figsize=(7, 1.0), constrained_layout=True)
for digit, ax in enumerate(axes):
    image = ax.imshow(
        weights[:, digit].reshape(28, 28),
        cmap="RdBu_r",
        norm=TwoSlopeNorm(vcenter=0, vmin=-limit, vmax=limit),
    )
    ax.set_title(str(digit), fontsize=8)
    ax.axis("off")
fig.colorbar(image, ax=axes, label="Weight", shrink=0.9)
fig.savefig(FIGURE_DIR / "logreg_weights.pdf")
plt.show()

#### Comparison with Scikit-Learn

Scikit-Learn minimizes $\frac{1}{2}\lVert W \rVert^2 + C \sum_i L_i$, a sum over the samples
rather than a mean. Dividing by $Cn$ shows that our $\lambda$ corresponds to
$C = 1/(2\lambda n)$, and $\lambda = 0$ to $C = \infty$.

In [ ]:
sklearn_logreg = SklearnLogisticRegression(
    C=1 / (2 * LOGREG_BEST_LAMBDA * len(x_fit)) if LOGREG_BEST_LAMBDA > 0 else np.inf,
    max_iter=1000,
    random_state=SEED,
)
sklearn_logreg.fit(x_fit, y_fit)

print(f"Ours:         {accuracy_score(y_val, logreg_pred):.4f}")
print(f"Scikit-Learn: {accuracy_score(y_val, sklearn_logreg.predict(x_val)):.4f}")

## Neural networks

We now add hidden layers between the input and the output. The network is built from the same
Dense and activation layers as in the regression notebook; only the output layer (10 logits)
and the cost function (the softmax cross entropy) change, and the predicted digit is the
largest logit.

Every network is trained with Adam and a batch size of 128, with two additions that a first,
fixed-length version of this experiment showed to be needed:

- **Early stopping.** Training for a fixed number of epochs let the validation cost rise while
  the training cost kept falling. We stop once the validation cost has not improved for
  `PATIENCE` epochs and restore the weights with the lowest validation cost.
- **A decaying learning rate.** A constant learning rate made the training cost noisy late in
  training. The learning rate decays exponentially, falling by a factor `LR_DECAY_FACTOR` over
  `MAX_EPOCHS` epochs.

Every configuration in the searches is trained the same way as the final network, so the
searches select what the final network actually uses.

> **LLM note:** The following cell was written with Claude assistance (level 4).

In [ ]:
# Weight initialization of the hidden layers, matched to their activation (Section 8.10)
INITIALIZATION = {Sigmoid: "xavier", ReLU: "he", LeakyReLU: "he"}

# Exponential decay rate that lowers the learning rate by LR_DECAY_FACTOR over MAX_EPOCHS
STEPS_PER_EPOCH = int(np.ceil(len(x_fit) / BATCH_SIZE))
LR_DECAY = np.log(LR_DECAY_FACTOR) / (MAX_EPOCHS * STEPS_PER_EPOCH)


def create_classifier(
    hidden_layers: list[int],
    optimizer: Optimizer,
    activation: type[Activation],
    lambda_: float = 0.0,
    learning_rate_schedule: Schedule = LR_SCHEDULE,
    *,
    rng: np.random.Generator,
) -> FFNN:
    """Build an FFNN for MNIST, with a softmax output trained with the cross entropy.

    Args:
        hidden_layers: Number of neurons in each hidden layer.
        optimizer: Fresh optimizer instance, copied for every weight matrix and bias.
        activation: The activation function class to use for the hidden layers.
        lambda_: The ℓ2 regularization strength, defaults to 0.0.
        learning_rate_schedule: How the learning rate decays during training.
        rng: Random number generator for the initial weights.

    Returns:
        The untrained network.
    """
    layers: list[Layer] = []
    n_inputs = x_fit.shape[1]
    for n_neurons in hidden_layers:
        layers += [Dense(n_inputs, n_neurons, INITIALIZATION[activation], rng=rng), activation()]
        n_inputs = n_neurons
    # The softmax is part of the loss, so the output layer is linear
    layers += [Dense(n_inputs, N_CLASSES, "xavier", rng=rng), Linear()]

    return FFNN(
        layers,
        CategoricalCrossEntropy(),
        optimizer,
        "l2",
        lambda_,
        SEED,
        learning_rate_schedule=learning_rate_schedule,
        lr_decay=LR_DECAY if learning_rate_schedule != "constant" else 0.0,
    )


def fit_classifier(
    hidden_layers: list[int],
    activation: type[Activation],
    learning_rate: float,
    lambda_: float = 0.0,
    learning_rate_schedule: Schedule = LR_SCHEDULE,
    patience: int = PATIENCE,
) -> tuple[FFNN, list[float]]:
    """Train a freshly initialized classifier on the training split.

    Training stops once the validation cost has not improved for ``patience`` epochs,
    and the weights from the epoch with the lowest validation cost are restored.
    """
    network = create_classifier(
        hidden_layers,
        OPTIMIZER(learning_rate=learning_rate),
        activation,
        lambda_,
        learning_rate_schedule,
        rng=np.random.default_rng(SEED),
    )
    # Too large learning rates can overflow; such runs simply get a low accuracy
    with np.errstate(over="ignore", invalid="ignore"):
        history = network.fit(
            x_fit,
            Y_fit,
            MAX_EPOCHS,
            BATCH_SIZE,
            validation_data=(x_val, Y_val),
            patience=patience,
        )
    return network, history


def predict_digits(network: FFNN, X: np.ndarray) -> np.ndarray:
    """Predicted digit of each sample, the class with the largest logit."""
    with np.errstate(over="ignore", invalid="ignore"):
        return np.argmax(network.predict(X), axis=1)


def run_summary(network: FFNN, history: list[float]) -> dict[str, float | int | None]:
    """Validation accuracy of a trained classifier and how long it trained."""
    return {
        "val_accuracy": accuracy_score(y_val, predict_digits(network, x_val)),
        "epochs_trained": len(history),
        "best_epoch": network.best_epoch,
    }


# Settings shared by both network searches, part of their cache keys
TRAINING_SETTINGS = {
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "lr_schedule": LR_SCHEDULE,
    "lr_decay_factor": LR_DECAY_FACTOR,
    "learning_rates": N_LEARNING_RATE,
}

### Activation function, depth and width

We first search over the activation function, the number of hidden layers and the number of
nodes per layer, without regularization. Since the best initial learning rate depends on the
activation and the architecture, every configuration is trained with each learning rate
$\gamma$, and we keep the best.

> **LLM note:** The following cell was written with Claude assistance (level 4). The search
space was chosen by the author.

In [ ]:
ACTIVATIONS_BY_NAME = {activation.__name__: activation for activation in ACTIVATION}


def architecture_search() -> pd.DataFrame:
    rows = []
    for activation, n_layers, n_nodes, learning_rate in product(
        ACTIVATION, N_HIDDEN_LAYERS, N_NODES_PER_LAYER, N_LEARNING_RATE
    ):
        network, history = fit_classifier([n_nodes] * n_layers, activation, learning_rate)
        rows.append(
            {
                "activation": activation.__name__,
                "n_layers": n_layers,
                "n_nodes": n_nodes,
                "learning_rate": learning_rate,
                **run_summary(network, history),
            }
        )
    return pd.DataFrame(rows)


architecture_grid = cached_search(
    "architectures",
    {
        **TRAINING_SETTINGS,
        "activations": list(ACTIVATIONS_BY_NAME),
        "n_hidden_layers": N_HIDDEN_LAYERS,
        "n_nodes_per_layer": N_NODES_PER_LAYER,
    },
    architecture_search,
)

# Best learning rate for each activation and architecture
best_per_architecture = architecture_grid.loc[
    architecture_grid.groupby(["activation", "n_layers", "n_nodes"])["val_accuracy"].idxmax()
]
best_architecture = best_per_architecture.loc[best_per_architecture["val_accuracy"].idxmax()]
BEST_ACTIVATION = ACTIVATIONS_BY_NAME[best_architecture["activation"]]
BEST_HIDDEN_LAYERS = [int(best_architecture["n_nodes"])] * int(best_architecture["n_layers"])
print(
    f"Best: {BEST_ACTIVATION.__name__}, hidden layers {BEST_HIDDEN_LAYERS}, "
    f"gamma = {best_architecture['learning_rate']:g}, "
    f"validation accuracy = {best_architecture['val_accuracy']:.4f}"
)
best_per_architecture.sort_values("val_accuracy", ascending=False).head(10).round(4)

The best validation accuracy of every architecture, with the learning rate that gave it.

In [ ]:
table_norm = accuracy_norm(100 * best_per_architecture["val_accuracy"].to_numpy())

fig, axes = plt.subplots(1, len(ACTIVATION), figsize=(7, 1.9), sharey=True, constrained_layout=True)
for ax, name in zip(axes, ACTIVATIONS_BY_NAME, strict=True):
    results = best_per_architecture[best_per_architecture["activation"] == name]
    accuracy, learning_rate = (
        results.pivot_table(index="n_layers", columns="n_nodes", values=column)
        .loc[N_HIDDEN_LAYERS, N_NODES_PER_LAYER]
        .to_numpy()
        for column in ["val_accuracy", "learning_rate"]
    )
    accuracy = 100 * accuracy

    image = ax.imshow(accuracy, cmap="Blues", norm=table_norm, aspect="auto")
    for (i, j), value in np.ndenumerate(accuracy):
        color = "white" if table_norm(value) > 0.6 else "#1f1e1d"
        ax.text(j, i - 0.12, f"{value:.1f}", ha="center", va="center", fontsize=7, color=color)
        label = rf"$\gamma=${power_label(learning_rate[i, j])}"
        ax.text(j, i + 0.22, label, ha="center", va="center", fontsize=5, color=color)
    ax.set_xticks(range(len(N_NODES_PER_LAYER)), N_NODES_PER_LAYER)
    ax.set_yticks(range(len(N_HIDDEN_LAYERS)), N_HIDDEN_LAYERS)
    ax.set_xlabel("Nodes per hidden layer")
    ax.set_title(name, fontsize=9)
    style_heatmap(ax)
axes[0].set_ylabel("Hidden layers")
fig.colorbar(image, ax=axes, label="Validation accuracy (percent)", shrink=0.9, extend="min")
fig.savefig(FIGURE_DIR / "ffnn_architectures.pdf")
plt.show()

The sensitivity to the learning rate: the best architecture of each activation at every
$\gamma$.

In [ ]:
plot_heatmap(
    100
    * architecture_grid.pivot_table(
        index="activation", columns="learning_rate", values="val_accuracy", aggfunc="max"
    )
    .loc[list(ACTIVATIONS_BY_NAME), N_LEARNING_RATE]
    .to_numpy(),
    x_labels=[power_label(v) for v in N_LEARNING_RATE],
    y_labels=list(ACTIVATIONS_BY_NAME),
    xlabel=r"Learning rate $\gamma$",
    ylabel=None,
    filename="ffnn_learning_rates.pdf",
    figsize=(3.3, 1.6),
    colorbar_label="Val. accuracy (percent)",
)

### Learning rate and regularization

For the best activation and architecture, we search the learning rate $\gamma$ against the
$\ell_2$ penalty $\lambda$. As for logistic regression, the penalty is a sum over all weights
while the cross entropy is a mean over the samples.

> **LLM note:** The following cell was written with Claude assistance (level 4). The search
space was chosen by the author.

In [ ]:
lambdas = sorted(REGULARIZATION_STRENGTH)


def regularization_search() -> pd.DataFrame:
    rows = []
    for learning_rate, lambda_ in product(N_LEARNING_RATE, lambdas):
        network, history = fit_classifier(
            BEST_HIDDEN_LAYERS, BEST_ACTIVATION, learning_rate, lambda_
        )
        rows.append(
            {"learning_rate": learning_rate, "lambda": lambda_, **run_summary(network, history)}
        )
    return pd.DataFrame(rows)


regularization_grid = cached_search(
    "regularization",
    {
        **TRAINING_SETTINGS,
        "activation": BEST_ACTIVATION.__name__,
        "hidden_layers": BEST_HIDDEN_LAYERS,
        "lambdas": lambdas,
    },
    regularization_search,
)

best = regularization_grid.loc[regularization_grid["val_accuracy"].idxmax()]
FFNN_LEARNING_RATE, FFNN_LAMBDA = best["learning_rate"], best["lambda"]
print(
    f"Best: gamma = {FFNN_LEARNING_RATE:g}, lambda = {FFNN_LAMBDA:g}, "
    f"validation accuracy = {best['val_accuracy']:.4f}, best epoch = {best['best_epoch']:.0f}"
)

In [ ]:
plot_heatmap(
    100
    * regularization_grid.pivot_table(
        index="lambda", columns="learning_rate", values="val_accuracy"
    )
    .loc[lambdas, N_LEARNING_RATE]
    .to_numpy(),
    x_labels=[power_label(v) for v in N_LEARNING_RATE],
    y_labels=[power_label(v) for v in lambdas],
    xlabel=r"Learning rate $\gamma$",
    ylabel=r"$\ell_2$ penalty $\lambda$",
    filename="ffnn_regularization.pdf",
    figsize=(3.3, 2.4),
)

### The selected network

We retrain the selected network with the same procedure as in the searches.

In [ ]:
ffnn, ffnn_history = fit_classifier(
    BEST_HIDDEN_LAYERS, BEST_ACTIVATION, FFNN_LEARNING_RATE, FFNN_LAMBDA
)
ffnn_pred = predict_digits(ffnn, x_val)
print(
    f"Stopped after {len(ffnn_history)} epochs, restored epoch {ffnn.best_epoch}; "
    f"validation accuracy: {accuracy_score(y_val, ffnn_pred):.4f}"
)

To see what the schedule and early stopping change, we also train it for all `MAX_EPOCHS`
epochs, with and without the schedule, and record the full learning curves. With
`patience = MAX_EPOCHS`, training never stops early, but the weights from the epoch with the
lowest validation cost are still restored.

> **LLM note:** The following cell was written with Claude assistance (level 4).

In [ ]:
full_runs = {
    schedule: fit_classifier(
        BEST_HIDDEN_LAYERS,
        BEST_ACTIVATION,
        FFNN_LEARNING_RATE,
        FFNN_LAMBDA,
        learning_rate_schedule=schedule,
        patience=MAX_EPOCHS,
    )
    for schedule in ["constant", LR_SCHEDULE]
}
plot_learning_curves(
    {
        schedule.capitalize(): (history, network.validation_history)
        for schedule, (network, history) in full_runs.items()
    },
    "ffnn_learning_curve.pdf",
    stopping_epoch=ffnn.best_epoch,
    log_scale=True,
)

pd.DataFrame.from_dict(
    {
        schedule: {
            "best_epoch": network.best_epoch,
            "min_val_cost": min(network.validation_history),
            "final_val_cost": network.validation_history[-1],
            "val_accuracy_at_best": accuracy_score(y_val, predict_digits(network, x_val)),
        }
        for schedule, (network, _) in full_runs.items()
    },
    orient="index",
).round(4)

The $F_1$ score of each digit, next to the logistic regression baseline.

In [ ]:
ffnn_metrics = class_metrics(y_val, ffnn_pred)
print(f"Macro-averaged F1 score: {ffnn_metrics['f1'].mean():.4f}")
pd.concat({"logreg_f1": logreg_metrics["f1"], "ffnn_f1": ffnn_metrics["f1"]}, axis=1).assign(
    change=lambda df: df["ffnn_f1"] - df["logreg_f1"]
).round(3)

### Comparison with Scikit-Learn

We train Scikit-Learn's `MLPClassifier` with the same architecture, optimizer, initial
learning rate and batch size, for as many epochs as our network trained before the weights it
kept. It adds $\frac{\alpha}{2 n_b}\lVert W \rVert^2$ to the mean loss of each minibatch of
size $n_b$, so our $\lambda$ corresponds to $\alpha = 2 \lambda n_b$. Its Adam solver has no
learning rate schedule, and it has no leaky ReLU; if that is the selected activation, we
compare against the ReLU.

> **LLM note:** The following cell was written with Claude assistance (level 4).

In [ ]:
SKLEARN_ACTIVATION = {Sigmoid: "logistic", ReLU: "relu", LeakyReLU: "relu"}

sklearn_mlp = MLPClassifier(
    hidden_layer_sizes=BEST_HIDDEN_LAYERS,
    activation=SKLEARN_ACTIVATION[BEST_ACTIVATION],
    solver="adam",
    alpha=2 * FFNN_LAMBDA * BATCH_SIZE,
    batch_size=BATCH_SIZE,
    learning_rate_init=FFNN_LEARNING_RATE,
    max_iter=ffnn.best_epoch,
    n_iter_no_change=ffnn.best_epoch,  # train for all max_iter epochs
    random_state=SEED,
)
# It warns that the fixed number of epochs is too few to converge, which is intended
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)
    sklearn_mlp.fit(x_fit, y_fit)

print(f"Ours:         {accuracy_score(y_val, ffnn_pred):.4f}")
print(f"Scikit-Learn: {accuracy_score(y_val, sklearn_mlp.predict(x_val)):.4f}")

## Final evaluation on the test set

The test set is used once, here, for the models selected on the validation data. We report the
accuracy and the macro-averaged $F_1$ score, and the confusion matrices of logistic regression
and the network.


In [ ]:
test_predictions = {
    "Majority class": np.full_like(y_test, majority_class),
    "Logistic regression (ours)": logistic_regression.predict(x_test),
    "Logistic regression (Scikit-Learn)": sklearn_logreg.predict(x_test),
    "FFNN (ours)": predict_digits(ffnn, x_test),
    "FFNN (Scikit-Learn)": sklearn_mlp.predict(x_test),
}
pd.DataFrame.from_dict(
    {
        model: {
            "test_accuracy": accuracy_score(y_test, y_pred),
            "test_macro_f1": macro_f1(y_test, y_pred),
        }
        for model, y_pred in test_predictions.items()
    },
    orient="index",
).round(4)

> **LLM note:** The following cell was written with Claude assistance (level 4).

In [ ]:
plot_confusion_matrix(
    y_test, test_predictions["Logistic regression (ours)"], "logreg_confusion_matrix.pdf"
)
plot_confusion_matrix(y_test, test_predictions["FFNN (ours)"], "ffnn_confusion_matrix.pdf")